In [1]:
from datasets import load_dataset

dataset = load_dataset("aclImdb_v1/aclImdb")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data)
print(test_data)

Resolving data files:   0%|          | 0/75003 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25002 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 150000
})
Dataset({
    features: ['text'],
    num_rows: 50000
})


In [7]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset["train"]
test_data = dataset["test"]

print(dataset)
print(dataset["train"].column_names)
print(dataset["test"].column_names)

print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
['text', 'label']
['text', 'label']
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United State

In [8]:
import pandas as pd

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(train_df.head())
print(test_df.head())

print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0
                                                text  label
0  I love sci-fi and am willing to put up with a ...      0
1  Worth the entertainment value of a rental, esp...      0
2  its a totally average film with a few semi-alr...      0
3  STAR RATING: ***** Saturday Night **** Friday ...      0
4  First off let me say, If you haven't enjoyed a...      0
label
0    12500
1    12500
Name: count, dtype: int64
label
0    12500
1    12500
Name: count, dtype: int64


In [9]:
X_train = train_df["text"].astype(str)
y_train = train_df["label"].astype(int)

X_test = test_df["text"].astype(str)
y_test = test_df["label"].astype(int)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 25000
Testing samples: 25000


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            sublinear_tf=True,
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            max_features=100000
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            C=2.0
        )
    )
])

In [11]:
print("Training model...")

pipeline.fit(X_train, y_train)

print("Training complete!")

Training model...
Training complete!


In [12]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

train_predictions = pipeline.predict(X_train)
test_predictions = pipeline.predict(X_test)

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Testing accuracy:  {test_accuracy:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Negative", "Positive"]
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))

Training accuracy: 0.9730
Testing accuracy:  0.9034

Classification report:
              precision    recall  f1-score   support

    Negative       0.91      0.90      0.90     12500
    Positive       0.90      0.91      0.90     12500

    accuracy                           0.90     25000
   macro avg       0.90      0.90      0.90     25000
weighted avg       0.90      0.90      0.90     25000


Confusion matrix:
[[11259  1241]
 [ 1174 11326]]


In [13]:
reviews = [
    "This movie was absolutely fantastic. I loved every minute of it.",
    "This was one of the worst movies I have ever watched.",
    "The acting was great and the story was very entertaining.",
    "Terrible movie. The plot was boring and the acting was awful."
]

predictions = pipeline.predict(reviews)

for review, prediction in zip(reviews, predictions):
    sentiment = "Positive (1)" if prediction == 1 else "Negative (0)"
    print(f"{sentiment}: {review}")

Positive (1): This movie was absolutely fantastic. I loved every minute of it.
Negative (0): This was one of the worst movies I have ever watched.
Positive (1): The acting was great and the story was very entertaining.
Negative (0): Terrible movie. The plot was boring and the acting was awful.


In [14]:
probabilities = pipeline.predict_proba(reviews)

for review, prediction, probability in zip(
    reviews,
    predictions,
    probabilities
):
    print("\nReview:", review)
    print("Prediction:", prediction)
    print("Negative probability:", probability[0])
    print("Positive probability:", probability[1])


Review: This movie was absolutely fantastic. I loved every minute of it.
Prediction: 1
Negative probability: 0.16907742499008271
Positive probability: 0.8309225750099173

Review: This was one of the worst movies I have ever watched.
Prediction: 0
Negative probability: 0.9912189348712398
Positive probability: 0.00878106512876022

Review: The acting was great and the story was very entertaining.
Prediction: 1
Negative probability: 0.017824267300483143
Positive probability: 0.9821757326995169

Review: Terrible movie. The plot was boring and the acting was awful.
Prediction: 0
Negative probability: 0.9991208485216523
Positive probability: 0.0008791514783477496


In [15]:
import joblib

MODEL_FILE = "skills_assessment.joblib"

joblib.dump(pipeline, MODEL_FILE)

print(f"Model saved as: {MODEL_FILE}")

Model saved as: skills_assessment.joblib


In [16]:
loaded_model = joblib.load("skills_assessment.joblib")

test_reviews = [
    "I absolutely loved this movie!",
    "This movie was terrible and boring."
]

predictions = loaded_model.predict(test_reviews)

for review, prediction in zip(test_reviews, predictions):
    print(
        f"{prediction} -> {review}"
    )

1 -> I absolutely loved this movie!
0 -> This movie was terrible and boring.
